<a href="https://colab.research.google.com/github/khovan123/cropstate/blob/main/notebooks/cropstate_colab_github_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CROPSTATE Colab Workflow

- Code clone từ GitHub; Dataset/KB/Results đọc-ghi trên Google Drive (`MyDrive/CROPSTATE_DATASET`, `CROPSTATE_KNOWLEDGE_BASE`, `CROPSTATE_RESULTS`).
- Trước khi train: `Runtime > Change runtime type` → chọn **GPU**.

## Thứ tự chạy (workflow chuẩn — không còn bước rename phẳng cũ)

**A. Chuẩn bị**
1. Mount Drive → 2. Clone/pull repo → 3. Cài deps → 4. Kiểm tra dataset/KB.

**B. Vision (leak-free)**
5. **Cell 4b — Encode parent_image_id** vào tên file (`p<NNN>_subset_overlap_<k>`). Bắt buộc, chạy MỘT lần trên Drive.
6. **Cell 6** — build `image_manifest_auto.csv` grouped (tuỳ chọn, cho audit). Cell 7/8 convert, cell 14/15 audit — đều tuỳ chọn.
7. **Cell 19** — train `vision_final` (tự build manifest **grouped** từ folder đã encode). Cell 20 fine-tune, 21 xem output, 22 predict.
8. **Cell 23–25** — novelty: retrain ordinal/focal + fixed-split multi-seed, bảng so sánh, và phân tích post-hoc (calibration/temporal/leakage).

**C. Retrieval / Knowledge base** (độc lập với vision)
9. Cell 9–13 — convert chunks, audit corpus, run/evaluate retrieval.

> ⚠️ Các cell rename phẳng `img001…` và repair theo thứ tự **đã bị gỡ bỏ** — chúng gây leakage. Nếu "Run all", mọi cell còn lại đều an toàn và nhất quán với parent-encoding.

In [1]:
# Sửa REPO_URL thành GitHub repo thật của bạn.
REPO_URL = "https://github.com/khovan123/cropstate"
PROJECT_DIR = "/content/CROPSTATE_Full_Research_Package"

DATA_ROOT = "/content/drive/MyDrive/CROPSTATE_DATASET"
KNOWLEDGE_ROOT = "/content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE"
RESULTS_ROOT = "/content/drive/MyDrive/CROPSTATE_RESULTS"
OUTPUT_DIR = f"{RESULTS_ROOT}/vision_final"

# Knowledge base: CHỈ dùng corpus IRRI (build từ raw_sources_irri/). Không dùng
# raw_sources hỗn hợp (PDF tiếng Việt/English) hay chunking cũ nữa.
KB_COMPLETE_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_irri_en.jsonl"
KB_NONRESTRICTED_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_irri_en_nonrestricted.jsonl"
RETRIEVAL_SCENARIOS = "data/retrieval_scenarios.csv"

print("REPO_URL:", REPO_URL)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("KNOWLEDGE_ROOT:", KNOWLEDGE_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

REPO_URL: https://github.com/khovan123/cropstate
PROJECT_DIR: /content/CROPSTATE_Full_Research_Package
DATA_ROOT: /content/drive/MyDrive/CROPSTATE_DATASET
KNOWLEDGE_ROOT: /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE
RESULTS_ROOT: /content/drive/MyDrive/CROPSTATE_RESULTS


## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Clone hoặc update repo từ GitHub

In [3]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/CROPSTATE_Full_Research_Package


## 3. Cài dependencies

In [4]:
!pip install -r requirements.txt
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 81.3 MB/s eta 0:00:00
Obtaining file:///content/CROPSTATE_Full_Research_Package
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cropstate-research (pyproject.toml) ... done
  Created wheel for cropstate-research: filename=cropstate_research-0.1.0-0.editable-py3-none-any.whl size=1383 sha256=61b5e4ae38b0f4c7356af69ef8800dbf66a01fa5a758c7f5d479262583918361
  Stored in directory: /tmp/pip-ephem-wheel-cache-xh3jvdun/wheels/37/37/8f/89550734e79d9e7fc68e041f688e3bc8f62ff3b93e9d2c762c
Successfully built cropstate-research


## 4. Kiểm tra dataset và knowledge base trên Drive

In [5]:
!ls -lah "{DATA_ROOT}"
!ls -lah "{KNOWLEDGE_ROOT}"

lrw------- 1 root root 0 Jul 10 18:14 /content/drive/MyDrive/CROPSTATE_DATASET -> /content/drive/.shortcut-targets-by-id/1iazxjNfESn9lm7fn-1L8ETOAtKSWVDYN/CROPSTATE_DATASET
lrw------- 1 root root 0 Jul 10 18:16 /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE -> /content/drive/.shortcut-targets-by-id/1rwtO-iHapZuwjFhRqwYcXm9RTyrvuHqS/CROPSTATE_KNOWLEDGE_BASE


Dataset nên có cấu trúc:

```text
CROPSTATE_DATASET/
  01_establishment/
  02_tillering/
  03_stem_booting/
  04_reproductive/
  05_grain_filling/
  06_ripening/
```

Knowledge base **chỉ dùng corpus IRRI**: folder `chunks/` chứa `rice_knowledge_irri_en.jsonl` (complete) và `rice_knowledge_irri_en_nonrestricted.jsonl`, cùng `raw_sources_irri/` (các trang IRRI Rice Knowledge Bank đã crawl). Không dùng `raw_sources/` hỗn hợp hay các corpus `rice_knowledge_complete/nonrestricted.jsonl` nữa.

## 4b. Encode parent_image_id thật vào tên file (chống leakage)

Gom các ảnh near-duplicate (patch chồng lấn cùng một cảnh) trong mỗi folder stage bằng perceptual hash, rồi đổi tên thành `p<NNN>_subset_overlap_<k>.jpg`. Nhờ đó `train_vision.py` (hàm `parent_image_id`) gom các patch cùng gốc vào **một** split → grouped split **leak-free**.

Trên bộ dữ liệu hiện tại (~98% ảnh độc lập), bước này chỉ gộp vài nhóm trùng thật (vd burst tillering 6 ảnh); phần còn lại mỗi ảnh là một parent riêng — đúng bản chất, không bịa cấu trúc.

**Thứ tự chạy:** cell **4b (encode)** → cell **6 (build manifest grouped)** → … → cell **19 (train)**.
- Cell **6** đã được cập nhật để **giữ** tên `p<NNN>_subset_overlap_<k>` (không rename phẳng nữa).
- Cell **16 (repair)** là legacy — bỏ qua.

**Lưu ý:**
- Đổi tên file trực tiếp trên Drive; mapping lưu ở `{DATA_ROOT}/parent_encoding_mapping.csv` để hoàn tác bằng `--undo`.
- Idempotent: chạy lại sẽ bỏ qua file đã có `_subset_overlap`.
- `parent_image_id` ở đây **suy ra từ clustering thị giác**, không phải metadata gốc — ghi rõ trong paper.

In [6]:
# Encode parent_image_id thật vào tên file trên Drive (idempotent).
# Gom near-duplicate trong mỗi folder stage -> p<NNN>_subset_overlap_<k>.<ext>
!PYTHONPATH=scripts/experiments python scripts/experiments/encode_parent_ids.py \
  --data-root "{DATA_ROOT}" --hamming-threshold 10 --apply

# Kiểm tra nhanh
!echo "Đã encode:" && find "{DATA_ROOT}"/0* -iname '*subset_overlap*' | wc -l
# Muốn hoàn tác: bỏ comment dòng dưới
# !PYTHONPATH=scripts/experiments python scripts/experiments/encode_parent_ids.py --data-root "{DATA_ROOT}" --undo

threshold=10  images=0  parents=0  multi-image parents=0  singletons=0  largest=0

Multi-image parent groups (candidate overlapping patches):

Renamed 0 files. Mapping (for --undo): /content/drive/MyDrive/CROPSTATE_DATASET/parent_encoding_mapping.csv
Đã encode:
609


## 5. Build manifest pilot từ folder dataset

In [7]:
!PYTHONPATH=src python scripts/build_stage_manifest.py \
  --data-root "{DATA_ROOT}" \
  --output data/stage_folder_manifest.csv

Wrote 609 rows to data/stage_folder_manifest.csv
macro_stage
tillering        160
ripening         148
grain_filling    125
establishment     72
stem_booting      63
reproductive      41


## 6. Build manifest từ tên file đã encode (grouped, KHÔNG rename phẳng)

Cell này đã được cập nhật: **không** còn đổi tên ảnh về `img001…` nữa (hành vi cũ đó chính là nguyên nhân gây leakage). Thay vào đó nó đọc tên đã encode ở **cell 4b** (`p<NNN>_subset_overlap_<k>`), lấy `parent_image_id = p<NNN>`, và ghi `image_manifest_auto.csv` với **split theo parent** (patch cùng gốc luôn cùng split).

Chạy **sau** cell 4b. Nếu chưa chạy cell 4b, cell sẽ cảnh báo và grouping sẽ suy biến về image-level.

> Lưu ý: cell **19 (train)** tự build manifest grouped riêng từ folder, nên `image_manifest_auto.csv` chủ yếu phục vụ các cell audit (7, 8, 14, 15). Không bắt buộc cho training.

In [8]:
# Build image_manifest_auto.csv từ tên file ĐÃ ENCODE ở cell 4b — KHÔNG rename phẳng.
# parent_image_id lấy từ tên: p<NNN>_subset_overlap_<k> -> parent = p<NNN>.
# Split theo PARENT (grouped, stratified theo stage) nên patch cùng gốc luôn cùng split -> không leak.
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_ROOT_PATH = Path(DATA_ROOT)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
STAGES = {
    "01_establishment": "S01 Establishment",
    "02_tillering": "S02 Tillering",
    "03_stem_booting": "S03 Stem/booting",
    "04_reproductive": "S04 Reproductive",
    "05_grain_filling": "S05 Grain filling",
    "06_ripening": "S06 Ripening",
}

rows = []
for folder_name, label in STAGES.items():
    folder = DATA_ROOT_PATH / folder_name
    if not folder.exists():
        print("Missing folder:", folder)
        continue
    for path in sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS):
        stem = path.stem
        parent = stem.split("_subset_overlap", 1)[0]  # p<NNN> nếu đã encode, ngược lại là stem gốc
        rows.append({
            "image_id": f"{folder_name}_{stem}",
            "file_name": f"{folder_name}/{path.name}",
            "drive_url": "", "source_name": "CROPSTATE_DATASET", "source_url": "",
            "license": "user_provided",
            "parent_image_id": f"{folder_name}:{parent}",
            "field_id": f"{folder_name}:{parent}",
            "capture_session": f"{folder_name}:{parent}",
            "capture_date": "", "region": "", "variety": "unknown",
            "annotator_1": "", "annotator_2": "",
            "final_label": label, "usable": True,
            "split": "", "review_status": "parent_encoded_grouped",
        })

manifest = pd.DataFrame(rows)
if manifest.empty:
    raise SystemExit("Không thấy ảnh — chạy cell 4b (encode) trước.")
if not manifest["file_name"].str.contains("_subset_overlap").any():
    print("⚠️  Chưa thấy tên _subset_overlap — bạn chưa chạy cell 4b (encode). Grouping sẽ = image-level.")

# Grouped stratified split THEO parent_image_id
groups = manifest.groupby("parent_image_id", as_index=False).agg(stage=("final_label", "first"))
sc = groups["stage"].value_counts()
trv, te = train_test_split(groups, test_size=0.15, random_state=42,
                           stratify=groups["stage"] if sc.min() >= 2 else None)
tc = trv["stage"].value_counts()
tr, va = train_test_split(trv, test_size=0.15 / 0.85, random_state=43,
                          stratify=trv["stage"] if tc.min() >= 2 else None)
split_map = {**{g: "train" for g in tr["parent_image_id"]},
             **{g: "validation" for g in va["parent_image_id"]},
             **{g: "test" for g in te["parent_image_id"]}}
manifest["split"] = manifest["parent_image_id"].map(split_map)

manifest_path = DATA_ROOT_PATH / "image_manifest_auto.csv"
manifest.to_csv(manifest_path, index=False)

straddle = manifest.groupby("parent_image_id")["split"].nunique()
print("Manifest:", manifest_path, "| rows:", len(manifest), "| parents:", manifest["parent_image_id"].nunique())
print("Parents xuyên split (phải = 0):", int((straddle > 1).sum()))
print("Splits:", manifest["split"].value_counts().to_dict())
print("Labels:", manifest["final_label"].value_counts().to_dict())

Manifest: /content/drive/MyDrive/CROPSTATE_DATASET/image_manifest_auto.csv | rows: 609 | parents: 566
Parents xuyên split (phải = 0): 0
Splits: {'train': 423, 'validation': 94, 'test': 92}
Labels: {'S02 Tillering': 160, 'S06 Ripening': 148, 'S05 Grain filling': 125, 'S01 Establishment': 72, 'S03 Stem/booting': 63, 'S04 Reproductive': 41}


## 7. Convert manifest auto thành manifest training

Cell này đọc `image_manifest_auto.csv` vừa tạo ở Drive và ghi ra `data/image_manifest.csv` trong repo Colab. Đây là manifest chính dùng để audit/train.


In [9]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --input "{DATA_ROOT}/image_manifest_auto.csv" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest.csv


Wrote training manifest: data/image_manifest.csv (609 rows)
Source mode: knowledge_manifest:/content/drive/MyDrive/CROPSTATE_DATASET/image_manifest_auto.csv
Excluded rows: 0
Exact duplicate image rows by sha256: 12


## 8. Convert image manifest từ folder stage

KB đã IRRI-only nên **không còn workbook image-manifest**. Cell này luôn **build manifest từ các folder stage** trong `DATA_ROOT` (parent_image_id suy từ tên `p<NNN>_subset_overlap_<k>`). Ghi ra `data/image_manifest_from_knowledge.csv` (không đè `data/image_manifest.csv` ở cell 7).

Vì không có sheet template, script sẽ in **`No Image_Manifest_Template found; building manifest from stage folders.`** — đây là hành vi **bình thường**, không phải lỗi.

> Nếu dataset trên Drive có `.ipynb_checkpoints/` bên trong folder stage, chúng bị quét lẫn → số dòng phình lên và báo trùng sha256. Xoá trước:
> `!find "{DATA_ROOT}" -type d -name .ipynb_checkpoints -exec rm -rf {{}} +`


In [10]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest_from_knowledge.csv

No Image_Manifest_Template found; building manifest from stage folders.
Wrote training manifest: data/image_manifest_from_knowledge.csv (609 rows)
Source mode: stage_folders
Excluded rows: 0
Exact duplicate image rows by sha256: 12


## 9. Audit corpus IRRI (complete)

Audit corpus IRRI đầy đủ (`rice_knowledge_irri_en.jsonl`) để xem coverage, restricted chunks và trạng thái production. Output lưu vào Drive results.

In [11]:
!mkdir -p "{RESULTS_ROOT}/retrieval"
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_COMPLETE_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_complete.json"


{
  "total_chunks": 173,
  "by_topic": {
    "disease_risk": 11,
    "general_crop_care": 16,
    "harvest_readiness": 18,
    "nutrient_management": 8,
    "pest_risk": 60,
    "residue_management": 16,
    "water_management": 16,
    "weed_management": 28
  },
  "by_facet": {
    "conditions": 48,
    "fertilizer": 10,
    "general": 73,
    "next_stage_action": 7,
    "pest_disease_prevention": 35
  },
  "by_source": {
    "IRRI_RKB_001": 8,
    "IRRI_RKB_002": 12,
    "IRRI_RKB_003": 5,
    "IRRI_RKB_004": 18,
    "IRRI_RKB_005": 8,
    "IRRI_RKB_006": 6,
    "IRRI_RKB_007": 25,
    "IRRI_RKB_008": 22,
    "IRRI_RKB_009": 17,
    "IRRI_RKB_010": 13,
    "IRRI_RKB_011": 9,
    "IRRI_RKB_012": 10,
    "IRRI_RKB_013": 11,
    "IRRI_RKB_014": 9
  },
  "by_review_status": {
    "machine_curated_pending_domain_review": 173
  },
  "stage_high_compatibility": {
    "establishment": 83,
    "tillering": 4,
    "stem_booting": 2,
    "reproductive": 5,
    "grain_filling": 1,
    "ripening":

## 10. Audit corpus IRRI (non-restricted)

Corpus IRRI đã loại chunk có restricted action (`rice_knowledge_irri_en_nonrestricted.jsonl`), phù hợp cho retrieval pilot.

In [12]:
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_NONRESTRICTED_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_nonrestricted.json"


{
  "total_chunks": 173,
  "by_topic": {
    "disease_risk": 11,
    "general_crop_care": 16,
    "harvest_readiness": 18,
    "nutrient_management": 8,
    "pest_risk": 60,
    "residue_management": 16,
    "water_management": 16,
    "weed_management": 28
  },
  "by_facet": {
    "conditions": 48,
    "fertilizer": 10,
    "general": 73,
    "next_stage_action": 7,
    "pest_disease_prevention": 35
  },
  "by_source": {
    "IRRI_RKB_001": 8,
    "IRRI_RKB_002": 12,
    "IRRI_RKB_003": 5,
    "IRRI_RKB_004": 18,
    "IRRI_RKB_005": 8,
    "IRRI_RKB_006": 6,
    "IRRI_RKB_007": 25,
    "IRRI_RKB_008": 22,
    "IRRI_RKB_009": 17,
    "IRRI_RKB_010": 13,
    "IRRI_RKB_011": 9,
    "IRRI_RKB_012": 10,
    "IRRI_RKB_013": 11,
    "IRRI_RKB_014": 9
  },
  "by_review_status": {
    "machine_curated_pending_domain_review": 173
  },
  "stage_high_compatibility": {
    "establishment": 83,
    "tillering": 4,
    "stem_booting": 2,
    "reproductive": 5,
    "grain_filling": 1,
    "ripening":

## 12. Run retrieval mẫu

Ví dụ: water management cho stage tillering. Script sẽ tải multilingual embedding model lần đầu chạy.

In [13]:
!PYTHONPATH=src python scripts/run_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --topic water_management \
  --stage tillering \
  --mode research \
  --top-k 5 \
  --output "{RESULTS_ROOT}/retrieval/water_tillering.json"


modules.json: 100% 229/229 [00:00<00:00, 882kB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 624kB/s]
README.md: 100% 3.89k/3.89k [00:00<00:00, 10.3MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 246kB/s]
config.json: 100% 645/645 [00:00<00:00, 4.00MB/s]
model.safetensors: 100% 471M/471M [00:05<00:00, 79.6MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 8481.69it/s]
tokenizer_config.json: 100% 526/526 [00:00<00:00, 3.37MB/s]
tokenizer.json: 100% 9.08M/9.08M [00:01<00:00, 6.59MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.19MB/s]
config.json: 100% 190/190 [00:00<00:00, 1.34MB/s]
{
  "topic": "water_management",
  "query": "Bằng chứng quản lý nước phù hợp với lúa ở giai đoạn Tillering, BBCH 20-29.",
  "belief": [
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0
  ],
  "confidence": 1.0,
  "mode": "research",
  "results": [
    {
      "rank": 1,
      "chunk_id": "IRRI_RKB_007_C0005",
      "text": "Large amounts of water can be lost dur

## 12b. Liệt kê chunk_id thật để tạo `data/retrieval_scenarios.csv`

Cell "Evaluate retrieval baselines" bên dưới cần file `data/retrieval_scenarios.csv` — đây là file bạn tự viết tay (ground truth đánh giá), không phải file do script tự sinh ra. Nếu file này chưa tồn tại, cell evaluate sẽ báo `FileNotFoundError`, đó là điều bình thường, không phải lỗi.

Cell này chỉ **liệt kê chunk_id thật** trong corpus của bạn theo từng topic/stage, để bạn có cơ sở chọn:

- `relevant_chunk_ids`: các chunk phù hợp giai đoạn đó (stage_compatibility ≥ 0.8);
- `incompatible_chunk_ids`: các chunk cùng topic nhưng không phù hợp giai đoạn đó (stage_compatibility = 0).

**Quan trọng:** danh sách này chỉ là gợi ý tự động dựa trên vector `stage_compatibility` sẵn có trong corpus. Không nên dùng thẳng làm ground truth khoa học cho paper, vì `stage_compatibility` cũng chính là tín hiệu mà bộ rerank dùng để xếp hạng — dùng lại nó làm nhãn đánh giá sẽ gây thiên lệch có lợi cho chính phương pháp đang được đánh giá (circular evaluation). Hãy dùng danh sách này làm điểm khởi đầu, sau đó tự rà soát/chỉnh sửa (hoặc nhờ người có chuyên môn nông nghiệp xác nhận) trước khi đưa vào `data/retrieval_scenarios.csv` thật.


In [14]:
import sys
sys.path.insert(0, "src")

from cropstate.knowledge import load_knowledge_chunks
from cropstate.constants import STAGE_NAMES

chunks = load_knowledge_chunks(KB_NONRESTRICTED_CORPUS, mode="research")
print(f"Total chunks loaded: {len(chunks)}\n")

topics = sorted({c.topic for c in chunks})
for topic in topics:
    topic_chunks = [c for c in chunks if c.topic == topic]
    print(f"=== topic: {topic} ({len(topic_chunks)} chunks) ===")
    for stage_index, stage in enumerate(STAGE_NAMES):
        relevant = [c for c in topic_chunks if c.stage_compatibility[stage_index] >= 0.5]
        incompatible = [c for c in topic_chunks if c.stage_compatibility[stage_index] == 0.0]
        if not relevant and not incompatible:
            continue
        print(f"  [{stage}]")
        if relevant:
            print(f"    relevant_chunk_ids candidates ({len(relevant)}):")
            for c in relevant[:8]:
                print(f"      {c.chunk_id}  -  {c.text[:70]}...")
        if incompatible:
            ids_preview = "|".join(c.chunk_id for c in incompatible[:8])
            print(f"    incompatible_chunk_ids candidates ({len(incompatible)}): {ids_preview}")
    print()

print("Copy các chunk_id phù hợp vào data/retrieval_scenarios.csv, cột relevant_chunk_ids")
print("và incompatible_chunk_ids cách nhau bằng dấu '|', ví dụ: kb_water_01|kb_water_07")


Total chunks loaded: 173

=== topic: disease_risk (11 chunks) ===
  [establishment]
    relevant_chunk_ids candidates (3):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
      IRRI_RKB_004_C0004  -  It must be grown, harvested, and processed correctly for best yield an...
      IRRI_RKB_004_C0017  -  The amount of moisture should be less than 14%, and preferably less th...
  [tillering]
    relevant_chunk_ids candidates (1):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
    incompatible_chunk_ids candidates (1): IRRI_RKB_004_C0017
  [stem_booting]
    incompatible_chunk_ids candidates (2): IRRI_RKB_004_C0004|IRRI_RKB_004_C0017
  [reproductive]
    incompatible_chunk_ids candidates (3): IRRI_RKB_003_C0005|IRRI_RKB_004_C0004|IRRI_RKB_004_C0017
  [grain_filling]
    incompatible_chunk_ids candidates (2): IRRI_RKB_003_C0005|IRRI_RKB_004_C0004
  [ripening]
    relevant_chunk_ids 

## 13. Evaluate retrieval baselines

Chỉ chạy cell này khi đã có `data/retrieval_scenarios.csv`. Metrics gồm P@k, R@k, nDCG@k, SIRR@k cho ungated, hard top-1, fixed-soft, adaptive-soft và oracle.

In [15]:
!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{RETRIEVAL_SCENARIOS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation.json"


Loading weights: 100% 199/199 [00:00<00:00, 4763.24it/s]
{
  "B0_ungated": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.1
  },
  "B1_query_expansion": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.0
  },
  "B2_hard_filter": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.1
  },
  "B3_fixed_soft": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.1
  },
  "P_adaptive_soft": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.1
  },
  "oracle_reference": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.0
  }
}


## 14. Audit manifest pilot từ folder

In [16]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum


Rows: 609

Class counts:
 macro_stage
tillering        160
ripening         148
grain_filling    125
establishment     72
stem_booting      63
reproductive      41
Name: count, dtype: int64

Split counts:
 split
unassigned    609
Name: count, dtype: int64

parent_image_id leakage groups: 0

field_id leakage groups: 0

capture_session leakage groups: 0

Unlabeled rows: 0

Missing image files: 0

Non-training labels in manifest: 0

Exact duplicate rows by sha256: 12
                   image_id  ...                                             sha256
34  p032_subset_overlap_000  ...  9132465f3183915eb26546f3effee7f936cc86e7d13fa6...
35  p032_subset_overlap_001  ...  9132465f3183915eb26546f3effee7f936cc86e7d13fa6...
36  p033_subset_overlap_000  ...  0c09041cadf8b91a0192b766f180c4b48c068711303054...
45  p033_subset_overlap_009  ...  9f10d3c0b2063f495aa51b6bc4bc44a1e2af8f2ab556aa...
47  p033_subset_overlap_011  ...  0c09041cadf8b91a0192b766f180c4b48c068711303054...
48  p033_subset_overlap_012

## 15. Audit manifest auto dùng để train

Đây là manifest được tạo từ cell rename và convert ở trên.

In [17]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum


Rows: 609

Class counts:
 macro_stage
tillering        160
ripening         148
grain_filling    125
establishment     72
stem_booting      63
reproductive      41
Name: count, dtype: int64

Split counts:
 split
train         423
validation     94
test           92
Name: count, dtype: int64

parent_image_id leakage groups: 0

field_id leakage groups: 0

capture_session leakage groups: 0

Unlabeled rows: 0

Missing image files: 0

Non-training labels in manifest: 0

Exact duplicate rows by sha256: 12
                                    image_id  ...                                             sha256
34  01_establishment_p032_subset_overlap_000  ...  9132465f3183915eb26546f3effee7f936cc86e7d13fa6...
35  01_establishment_p032_subset_overlap_001  ...  9132465f3183915eb26546f3effee7f936cc86e7d13fa6...
36  01_establishment_p033_subset_overlap_000  ...  0c09041cadf8b91a0192b766f180c4b48c068711303054...
45  01_establishment_p033_subset_overlap_009  ...  9f10d3c0b2063f495aa51b6bc4bc44a1e2af8f2a

## 19. Train lần đầu vào `vision_final`

Chỉ dùng một output folder duy nhất. Script tự tạo `OUTPUT_DIR/manifest.csv` từ các stage folder trong `DATA_ROOT`, rồi lưu checkpoint và test metrics vào cùng folder.

In [18]:
# Vision final. Split LEAK-FREE tự động: leak_hamming_threshold (configs/vision.yaml)
# gom near-duplicate xuyên stage vào cùng 1 split -> leakage_audit báo 0 cặp rò rỉ.
# --balanced-sampler: WeightedRandomSampler cân bằng lớp, nâng recall lớp hiếm (reproductive).
!PYTHONPATH=src python scripts/train_vision.py \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --balanced-sampler \
  --output "{OUTPUT_DIR}"

model.safetensors: 100% 46.8M/46.8M [00:01<00:00, 28.9MB/s]
1 {'accuracy': 0.24705882352941178, 'macro_f1': 0.19151709401709402, 'masd': 1.7529411764705882, 'confusion_matrix': [[1, 6, 0, 0, 0, 0], [0, 11, 1, 9, 0, 1], [0, 7, 0, 2, 0, 0], [0, 2, 0, 3, 0, 1], [0, 15, 0, 2, 0, 2], [0, 12, 0, 4, 0, 6]], 'loss': 1.7328282992045085, 'brier': 0.8139430846665259, 'ece': 0.06501266009667342, 'per_class_recall': {'establishment': 0.14285714285714285, 'tillering': 0.5, 'stem_booting': 0.0, 'reproductive': 0.5, 'grain_filling': 0.0, 'ripening': 0.2727272727272727}}
2 {'accuracy': 0.4470588235294118, 'macro_f1': 0.3435142515787677, 'masd': 1.1176470588235294, 'confusion_matrix': [[3, 4, 0, 0, 0, 0], [0, 17, 1, 1, 0, 3], [0, 5, 0, 2, 1, 1], [0, 1, 0, 0, 3, 2], [0, 8, 1, 0, 7, 3], [0, 5, 0, 0, 6, 11]], 'loss': 1.6876825888951619, 'brier': 0.7917970067597369, 'ece': 0.24392017844845265, 'per_class_recall': {'establishment': 0.42857142857142855, 'tillering': 0.7727272727272727, 'stem_booting': 0.0, 'r

## 20. Fine-tune tiếp trong cùng `vision_final`

Dùng cell này cho lần chạy tiếp theo. Checkpoint cũ được đọc từ `OUTPUT_DIR/best_checkpoint.pt`; nếu validation tốt hơn, checkpoint mới ghi đè vào đúng file đó.

In [19]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{OUTPUT_DIR}/manifest.csv" \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --resume-checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --freeze-backbone-epochs 0 \
  --learning-rate 0.0001 \
  --backbone-learning-rate 0.00001 \
  --output "{OUTPUT_DIR}"

31 {'accuracy': 0.7764705882352941, 'macro_f1': 0.7361662635770392, 'masd': 0.29411764705882354, 'confusion_matrix': [[6, 1, 0, 0, 0, 0], [1, 19, 2, 0, 0, 0], [0, 2, 6, 1, 0, 0], [0, 1, 1, 3, 0, 1], [0, 2, 0, 2, 11, 4], [0, 0, 0, 0, 1, 21]], 'loss': 0.5945588648319244, 'brier': 0.3031722345652091, 'ece': 0.16769843346932356, 'per_class_recall': {'establishment': 0.8571428571428571, 'tillering': 0.8636363636363636, 'stem_booting': 0.6666666666666666, 'reproductive': 0.5, 'grain_filling': 0.5789473684210527, 'ripening': 0.9545454545454546}}
32 {'accuracy': 0.8352941176470589, 'macro_f1': 0.8008511617207269, 'masd': 0.2, 'confusion_matrix': [[6, 1, 0, 0, 0, 0], [1, 19, 2, 0, 0, 0], [0, 2, 6, 1, 0, 0], [0, 0, 1, 4, 0, 1], [0, 1, 0, 1, 15, 2], [0, 0, 0, 0, 1, 21]], 'loss': 0.5782981912295023, 'brier': 0.2899054201580242, 'ece': 0.16321070790290831, 'per_class_recall': {'establishment': 0.8571428571428571, 'tillering': 0.8636363636363636, 'stem_booting': 0.6666666666666666, 'reproductive': 0

## 21. Xem output đã lưu trên Drive

Folder này chỉ nên có `best_checkpoint.pt`, `history.json`, `class_counts.json`, `manifest.csv`, và `test_metrics.json`.

In [20]:
!ls -lah "{OUTPUT_DIR}"

total 43M
-rw------- 1 root root  43M Jul 10 18:32 best_checkpoint.pt
-rw------- 1 root root  471 Jul 10 18:37 class_counts.json
-rw------- 1 root root 140K Jul 10 18:37 history.json
-rw------- 1 root root 123K Jul 10 18:33 manifest.csv
-rw------- 1 root root 2.1K Jul 10 18:37 test_metrics.json


## 22. Test một ảnh upload từ máy

In [21]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

Saving cay-lua.png to cay-lua.png
Uploaded: cay-lua.png


In [22]:
!PYTHONPATH=src python scripts/predict_image.py \
  --checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --image "{image_path}"

{
  "image_path": "cay-lua.png",
  "predicted_id": 0,
  "predicted_stage": "establishment",
  "predicted_stage_display": "Establishment",
  "confidence": 0.3015691041946411,
  "stage_belief": {
    "establishment": 0.3015691041946411,
    "tillering": 0.13406504690647125,
    "stem_booting": 0.03985988721251488,
    "reproductive": 0.1846974939107895,
    "grain_filling": 0.07227920740842819,
    "ripening": 0.26752933859825134
  }
}


## 23. Novelty experiments — ordinal/focal loss + fixed-split multi-seed (GPU)

Các thí nghiệm tăng novelty cho paper. Tất cả retrain ResNet-18 trên **cùng một split cố định** với `vision_final` (`OUTPUT_DIR/manifest.csv`) để so sánh trực tiếp với baseline CE:

- **ordinal** (Tier A#2): loss = CE + λ·(E[stage] − stage_thật)² → phạt mạnh lỗi cách xa giai đoạn, giảm MASD và lỗi non-adjacent.
- **focal** (Tier C#6): giảm trọng số mẫu dễ, giúp lớp hiếm (reproductive).
- **seed 7 / seed 123** (Tier C#9): cùng split cố định, chỉ đổi seed → tách phương sai khởi tạo khỏi phương sai split.

**Trước khi chạy:**
1. Đã `git push` code mới lên GitHub: `scripts/train_vision.py` (thêm `--loss`), `src/cropstate/losses.py`, `scripts/experiments/`.
2. Đã chạy lại **cell 2 (git pull)** để Colab có code mới.
3. Đã chạy **cell 19** (train `vision_final`) để có split cố định.

Colab có GPU nên **không** cần `CROPSTATE_FORCE_CPU`. Mỗi lần train ~vài phút trên GPU.

In [23]:
NOVELTY_DIR = f"{RESULTS_ROOT}/novelty"
FIXED_MANIFEST = f"{OUTPUT_DIR}/manifest.csv"  # split cố định từ baseline CE (cell 19)

import os
os.makedirs(NOVELTY_DIR, exist_ok=True)
assert os.path.exists(FIXED_MANIFEST), (
    f"Thiếu {FIXED_MANIFEST} — chạy cell 19 (train vision_final) trước để tạo split cố định."
)
print("Fixed split :", FIXED_MANIFEST)
print("Novelty dir :", NOVELTY_DIR)

Fixed split : /content/drive/MyDrive/CROPSTATE_RESULTS/vision_final/manifest.csv
Novelty dir : /content/drive/MyDrive/CROPSTATE_RESULTS/novelty


In [24]:
# Tier A#2 — ordinal loss (CE + λ·(E[stage] − stage_thật)²)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss ordinal \
  --output "{NOVELTY_DIR}/resnet18_ordinal"

1 {'accuracy': 0.21176470588235294, 'macro_f1': 0.20978896516653953, 'masd': 1.4470588235294117, 'confusion_matrix': [[2, 5, 0, 0, 0, 0], [0, 5, 0, 16, 0, 1], [0, 3, 0, 4, 0, 2], [0, 2, 0, 2, 1, 1], [0, 8, 0, 6, 1, 4], [0, 1, 0, 11, 2, 8]], 'loss': 3.3784337838490806, 'brier': 0.8130511303185839, 'ece': 0.05432894072111916, 'per_class_recall': {'establishment': 0.2857142857142857, 'tillering': 0.22727272727272727, 'stem_booting': 0.0, 'reproductive': 0.3333333333333333, 'grain_filling': 0.05263157894736842, 'ripening': 0.36363636363636365}}
2 {'accuracy': 0.3176470588235294, 'macro_f1': 0.26524943310657595, 'masd': 1.423529411764706, 'confusion_matrix': [[3, 4, 0, 0, 0, 0], [0, 4, 1, 1, 5, 11], [0, 1, 0, 1, 1, 6], [0, 0, 0, 0, 3, 3], [0, 1, 0, 0, 10, 8], [0, 0, 0, 1, 11, 10]], 'loss': 3.228521188100179, 'brier': 0.7914407637686273, 'ece': 0.10839440366801092, 'per_class_recall': {'establishment': 0.42857142857142855, 'tillering': 0.18181818181818182, 'stem_booting': 0.0, 'reproductive'

In [25]:
# Tier C#6 — focal loss (giúp lớp hiếm reproductive)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss focal \
  --output "{NOVELTY_DIR}/resnet18_focal"

1 {'accuracy': 0.24705882352941178, 'macro_f1': 0.2812215687028281, 'masd': 1.4, 'confusion_matrix': [[5, 2, 0, 0, 0, 0], [0, 5, 1, 16, 0, 0], [0, 4, 0, 5, 0, 0], [0, 2, 0, 4, 0, 0], [0, 9, 1, 9, 0, 0], [0, 2, 0, 12, 1, 7]], 'loss': 1.1401174863179524, 'brier': 0.818315706497181, 'ece': 0.047913842692094685, 'per_class_recall': {'establishment': 0.7142857142857143, 'tillering': 0.22727272727272727, 'stem_booting': 0.0, 'reproductive': 0.6666666666666666, 'grain_filling': 0.0, 'ripening': 0.3181818181818182}}
2 {'accuracy': 0.4588235294117647, 'macro_f1': 0.4552934258816612, 'masd': 0.8705882352941177, 'confusion_matrix': [[6, 1, 0, 0, 0, 0], [0, 8, 3, 6, 3, 2], [0, 1, 3, 3, 2, 0], [0, 0, 1, 0, 3, 2], [0, 0, 1, 1, 14, 3], [0, 1, 0, 2, 11, 8]], 'loss': 1.0965914726257324, 'brier': 0.8002752447805657, 'ece': 0.2616083483485614, 'per_class_recall': {'establishment': 0.8571428571428571, 'tillering': 0.36363636363636365, 'stem_booting': 0.3333333333333333, 'reproductive': 0.0, 'grain_filling

In [30]:
# Tier C#9 — fixed-split, seed 7
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 7 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed7"

1 {'accuracy': 0.3058823529411765, 'macro_f1': 0.18786081121607437, 'masd': 1.6941176470588235, 'confusion_matrix': [[1, 2, 3, 0, 0, 1], [0, 19, 2, 0, 1, 0], [0, 8, 1, 0, 0, 0], [0, 2, 1, 0, 1, 2], [1, 14, 1, 1, 0, 2], [0, 11, 2, 3, 1, 5]], 'loss': 1.7548340559005737, 'brier': 0.8181693741953318, 'ece': 0.16067091542131762, 'per_class_recall': {'establishment': 0.14285714285714285, 'tillering': 0.8636363636363636, 'stem_booting': 0.1111111111111111, 'reproductive': 0.0, 'grain_filling': 0.0, 'ripening': 0.22727272727272727}}
2 {'accuracy': 0.38823529411764707, 'macro_f1': 0.28413895472719003, 'masd': 1.4588235294117646, 'confusion_matrix': [[1, 3, 1, 0, 1, 1], [0, 18, 3, 1, 0, 0], [0, 5, 3, 0, 0, 1], [0, 1, 0, 0, 2, 3], [1, 12, 0, 1, 2, 3], [0, 9, 1, 1, 2, 9]], 'loss': 1.7237796386082966, 'brier': 0.803035644125495, 'ece': 0.18921889662742614, 'per_class_recall': {'establishment': 0.14285714285714285, 'tillering': 0.8181818181818182, 'stem_booting': 0.3333333333333333, 'reproductive': 

In [27]:
# Tier C#9 — fixed-split, seed 123
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 123 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed123"

1 {'accuracy': 0.38823529411764707, 'macro_f1': 0.27185869685869685, 'masd': 1.411764705882353, 'confusion_matrix': [[1, 3, 1, 0, 0, 2], [1, 10, 5, 0, 0, 6], [1, 4, 2, 0, 0, 2], [1, 4, 0, 0, 0, 1], [0, 7, 3, 0, 3, 6], [1, 2, 0, 2, 0, 17]], 'loss': 1.7232634623845418, 'brier': 0.8100768644976034, 'ece': 0.19357079810955946, 'per_class_recall': {'establishment': 0.14285714285714285, 'tillering': 0.45454545454545453, 'stem_booting': 0.2222222222222222, 'reproductive': 0.0, 'grain_filling': 0.15789473684210525, 'ripening': 0.7727272727272727}}
2 {'accuracy': 0.4470588235294118, 'macro_f1': 0.313763154361723, 'masd': 1.1529411764705881, 'confusion_matrix': [[0, 6, 1, 0, 0, 0], [0, 10, 5, 0, 0, 7], [0, 4, 4, 0, 0, 1], [0, 4, 0, 0, 1, 1], [1, 4, 2, 0, 7, 5], [0, 3, 0, 0, 2, 17]], 'loss': 1.7032945553461711, 'brier': 0.7980139531297551, 'ece': 0.2480111607733895, 'per_class_recall': {'establishment': 0.0, 'tillering': 0.45454545454545453, 'stem_booting': 0.4444444444444444, 'reproductive': 0.0

## 24. So sánh biến thể novelty với baseline CE

Đọc `test_metrics.json` của từng biến thể (cùng split cố định) và in bảng: accuracy, macro-F1, MASD, ECE, recall lớp reproductive. Kỳ vọng: **ordinal** giảm MASD và lỗi non-adjacent; **focal** tăng recall reproductive; **seed7/seed123** cho dải phương sai khởi tạo trên split cố định.

In [28]:
import json, os
import pandas as pd

variants = {
    "CE baseline (vision_final)": OUTPUT_DIR,
    "ordinal (A#2)":              f"{NOVELTY_DIR}/resnet18_ordinal",
    "focal (C#6)":                f"{NOVELTY_DIR}/resnet18_focal",
    "CE seed7 (C#9)":             f"{NOVELTY_DIR}/resnet18_ce_seed7",
    "CE seed123 (C#9)":           f"{NOVELTY_DIR}/resnet18_ce_seed123",
}
rows = []
for name, d in variants.items():
    p = os.path.join(d, "test_metrics.json")
    if not os.path.exists(p):
        print("bỏ qua (chưa train):", name)
        continue
    m = json.load(open(p))
    rows.append({
        "variant": name,
        "accuracy": round(m.get("accuracy", float("nan")), 3),
        "macro_f1": round(m.get("macro_f1", float("nan")), 3),
        "masd": round(m.get("masd", float("nan")), 3),
        "ece": round(m.get("ece", float("nan")), 3),
        "reproductive_recall": round((m.get("per_class_recall") or {}).get("reproductive", float("nan")), 3),
    })

df = pd.DataFrame(rows)
df.to_csv(f"{NOVELTY_DIR}/retrain_comparison.csv", index=False)
print(df.to_string(index=False))
print("\nĐã lưu:", f"{NOVELTY_DIR}/retrain_comparison.csv")

                   variant  accuracy  macro_f1  masd   ece  reproductive_recall
CE baseline (vision_final)     0.824     0.775 0.275 0.148                0.333
             ordinal (A#2)     0.780     0.702 0.385 0.162                0.167
               focal (C#6)     0.791     0.765 0.297 0.344                0.667
            CE seed7 (C#9)     0.802     0.765 0.330 0.379                0.500
          CE seed123 (C#9)     0.824     0.815 0.231 0.169                0.667

Đã lưu: /content/drive/MyDrive/CROPSTATE_RESULTS/novelty/retrain_comparison.csv


## 25. (Tuỳ chọn) Phân tích post-hoc trên Colab → lưu vào Drive

Export logits từ các checkpoint có sẵn rồi chạy calibration (temperature scaling, Tier A#3), temporal fusion (Tier B#5) và leakage audit (Tier A#1). Cần checkpoint `vision_final` (và nếu có `vision_small_cnn`, `vision_efficientnet_b0`) trong `RESULTS_ROOT`. Kết quả JSON lưu vào `RESULTS_ROOT/novelty` để kéo về.

In [29]:
CKPTS = {
    "resnet18": f"{OUTPUT_DIR}/best_checkpoint.pt",
    "small_cnn": f"{RESULTS_ROOT}/vision_small_cnn/best_checkpoint.pt",
    "efficientnet_b0": f"{RESULTS_ROOT}/vision_efficientnet_b0/best_checkpoint.pt",
}
import os
ck = " ".join(f'--checkpoint {n}={p}' for n, p in CKPTS.items() if os.path.exists(p))
print("Checkpoints dùng được:", ck or "(chỉ resnet18)")

!PYTHONPATH=src python scripts/experiments/export_logits.py {ck} \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --output-dir "{NOVELTY_DIR}/logits"

# Calibration (temperature scaling) — torch-free, cắt ECE
!PYTHONPATH=src python scripts/experiments/calibration.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" \
  --output "{NOVELTY_DIR}/calibration.json"

# Temporal phenology fusion (Tier B#5)
!PYTHONPATH=src python scripts/experiments/temporal_fusion.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --output "{NOVELTY_DIR}/temporal_fusion.json"

# Near-duplicate / leakage audit (Tier A#1)
!PYTHONPATH=src python scripts/experiments/leakage_audit.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" --hamming-threshold 10 \
  --output "{NOVELTY_DIR}/leakage_audit.json"

print("Xong. Kết quả trong", NOVELTY_DIR)

Checkpoints dùng được: --checkpoint resnet18=/content/drive/MyDrive/CROPSTATE_RESULTS/vision_final/best_checkpoint.pt
[export] resnet18: validation=85, test=91
[resnet18] T=0.557  ECE 0.148 -> 0.105   Brier 0.305 -> 0.286   NLL 0.657 -> 0.648
{
  "single_frame_argmax": {
    "accuracy": 0.8241758241758241,
    "masd": 0.27472527472527475,
    "non_adjacent_errors": 7,
    "adjacent_errors": 9
  },
  "temporal_fused_mean_over_20_orderings": {
    "accuracy": 0.7857142857142858,
    "masd": 0.2302197802197802,
    "non_adjacent_errors": 1.35,
    "adjacent_errors": 18.15
  },
  "non_adjacent_error_reduction": 5.65,
  "masd_reduction": 0.04450549450549454
}
Traceback (most recent call last):
  File "/content/CROPSTATE_Full_Research_Package/scripts/experiments/leakage_audit.py", line 142, in <module>
    main()
  File "/content/CROPSTATE_Full_Research_Package/scripts/experiments/leakage_audit.py", line 136, in main
    df.to_csv(args.grouped_manifest, index=False)
  File "/usr/local/lib/py

## 12. Retrieval đóng vòng từ belief thật (robust)

`evaluate_retrieval` ở mục trước chỉ có 2 kịch bản minh hoạ. Ở đây sinh kịch bản
từ **belief 6 lớp thật của mọi ảnh test** (logits vừa export), gán relevance
tự động theo `stage_compatibility` của từng chunk IRRI (không cần chuyên gia gán tay).
Chỉ giữ cặp (topic, stage) mà corpus trả lời được (≥3 relevant & ≥3 incompatible),
nên số liệu không còn suy biến về 0 như bản 2-kịch-bản.


In [ ]:
# Sinh kịch bản từ belief thật + corpus IRRI, rồi đánh giá retrieval (đóng vòng).
!PYTHONPATH=src python scripts/experiments/build_belief_scenarios.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --calibration "{NOVELTY_DIR}/calibration.json" \
  --output "{NOVELTY_DIR}/belief_scenarios.csv"

!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{NOVELTY_DIR}/belief_scenarios.csv" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation_belief.json"

## Ghi chú

- GitHub chỉ lưu code và metadata nhỏ.
- Google Drive lưu dataset, knowledge base, checkpoint và kết quả training.
- Không commit các file `.pt` lên GitHub.